# Binding site prediction — gradient boosting on hand crafted features

**Approach**
1. Parse the PDB and compute **66 per-residue features** — amino-acid identity
2. Train a `HistGradientBoostingClassifier` on all 16379 training proteins.
3. Pick the probability threshold that maximises the flat Jaccard on a
   protein level held out split.
4. **Post-process** with spatial dilation + connected component pruning.

In [1]:
import os, sys, time, pickle, zipfile
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import jaccard_score, roc_auc_score, average_precision_score

# Paths are relative to the challenge root; run `jupyter` from there (or adjust).
ROOT       = os.path.abspath("..") if os.path.basename(os.getcwd()) == "01_gbm_handcrafted" else os.getcwd()
DATA_DIR   = os.path.join(ROOT, "data")
TRAIN_DIR  = os.path.join(DATA_DIR, "train")          # 16 379 *_protein.pdb
TEST_DIR   = os.path.join(DATA_DIR, "test")           #  3 064 *_protein.pdb
TRAIN_CSV  = os.path.join(DATA_DIR, "train.csv")
CACHE_DIR  = os.path.join(ROOT, "01_gbm_handcrafted", "cache")
MODEL_DIR  = os.path.join(ROOT, "01_gbm_handcrafted", "model")
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(MODEL_DIR, exist_ok=True)

SEED = 20250618
rng  = np.random.default_rng(SEED)

In [2]:
labels = pd.read_csv(TRAIN_CSV)
print(labels.shape, "training proteins")
display(labels.head(3))

def read_pdb_bytes(folder, pid):
    # raw PDB bytes for one protein, from a directory (or a .zip archive)
    name = f"{pid}_protein.pdb"
    if os.path.isdir(folder):
        path = os.path.join(folder, name)
        return open(path, "rb").read() if os.path.exists(path) else None
    with zipfile.ZipFile(folder) as zf:               # fallback: train.zip/test.zip
        try:    return zf.read(name)
        except KeyError: return None

sample = read_pdb_bytes(TRAIN_DIR, 0).decode("ascii", "ignore").splitlines()
print(f"\nprotein 0 has {len(sample)} PDB lines; first ATOM records:")
for line in [l for l in sample if l.startswith("ATOM")][:3]:
    print("  ", line)
print("\nbinding residues for protein 0:", labels.loc[0, "resid"][:80], "...")

(16379, 2) training proteins


,id,resid
0,0,A_24 A_25 A_26 A_27 A_28 A_29 A_30 A_31 A_32 A...
1,1,A_14 A_15 A_16 A_28 A_30 A_31 A_54 A_87 A_89 A...
2,2,A_249 A_255 A_259 A_268 A_270 A_277 A_278 A_27...



protein 0 has 8078 PDB lines; first ATOM records:
   ATOM      1  N   ARG A   5      31.378  57.568  22.411  1.00 53.76           N  
   ATOM      2  HN1 ARG A   5      31.219  57.855  23.398  1.00  0.00           H  
   ATOM      3  HN2 ARG A   5      32.361  57.774  22.143  1.00  0.00           H  

binding residues for protein 0: A_24 A_25 A_26 A_27 A_28 A_29 A_30 A_31 A_32 A_33 A_34 A_35 A_37 A_38 A_43 A_104 ...


###   Amino-acid tables and the PDB parser

In [3]:
AA3_TO_1 = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
    "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P",
    "SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M","SEC":"C","PYL":"K",
    "HID":"H","HIE":"H","HIP":"H","CYX":"C","CYM":"C","ASH":"D","GLH":"E","LYN":"K"}
AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"
AA_INDEX = {a: i for i, a in enumerate(AA_ORDER)}

KD  = {"A":1.8,"C":2.5,"D":-3.5,"E":-3.5,"F":2.8,"G":-0.4,"H":-3.2,"I":4.5,"K":-3.9,
       "L":3.8,"M":1.9,"N":-3.5,"P":-1.6,"Q":-3.5,"R":-4.5,"S":-0.8,"T":-0.7,"V":4.2,"W":-0.9,"Y":-1.3}
VOL = {"A":88.6,"C":108.5,"D":111.1,"E":138.4,"F":189.9,"G":60.1,"H":153.2,"I":166.7,
       "K":168.6,"L":166.7,"M":162.9,"N":114.1,"P":112.7,"Q":143.8,"R":173.4,"S":89.0,
       "T":116.1,"V":140.0,"W":227.8,"Y":193.6}
CHARGE = {"A":0,"C":0,"D":-1,"E":-1,"F":0,"G":0,"H":0.1,"I":0,"K":1,"L":0,"M":0,"N":0,
          "P":0,"Q":0,"R":1,"S":0,"T":0,"V":0,"W":0,"Y":0}
POLAR=set("STNQHYCDEKR"); AROMATIC=set("FWYH"); ALIPHATIC=set("AVLIMG")
HBOND_DONOR=set("RKQNHWSTY"); HBOND_ACCEPTOR=set("DENQHSTY")

PROP_NAMES = ["kd","vol","charge","polar","aromatic","aliphatic","donor","acceptor","glycine"]
GEOM_FEATS = ["n6","n8","n10","n12","n15","d_centroid","d_centroid_norm","depth_proxy",
    "mean_nn_dist","std_nn_dist","hse_up","hse_down","hse_ratio","ca_z_norm",
    "bfactor_mean","bfactor_std","side_chain_extent","pocket_open_space",
    "pocket_concavity","sa_proxy","anisotropy","rel_burial"]
NEIGH_AGG_PROPS = ["kd","vol","charge","polar","aromatic","donor","acceptor"]
NEIGH_RADII = [8.0, 12.0]
NEIGH_FEATS = [f"n8_mean_{p}" for p in NEIGH_AGG_PROPS] + [f"n12_mean_{p}" for p in NEIGH_AGG_PROPS]
FEATURE_COLS = ([f"aa_{a}" for a in AA_ORDER] + ["aa_unknown"]
                + [f"prop_{n}" for n in PROP_NAMES] + GEOM_FEATS + NEIGH_FEATS)
NUM_FEATURES = len(FEATURE_COLS)
assert NUM_FEATURES == 66

def parse_pdb_atoms(pdb_bytes):
    # group ATOM records by residue; keep Ca, Cb, side-chain atoms and B-factors
    residues, order = {}, []
    text = pdb_bytes.decode("ascii", "ignore") if isinstance(pdb_bytes,(bytes,bytearray)) else pdb_bytes
    for line in text.splitlines():
        if len(line) < 54 or line[:6] != "ATOM  ":
            continue
        atom = line[12:16].strip()
        if line[16:17] not in (" ", "A", ""):          # skip alt-loc B/C/
            continue
        chain = line[21:22]
        try:
            resnum = int(line[22:26])
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
        except ValueError:
            continue
        icode = line[26:27]; icode = "" if icode == " " else icode
        try:    bfac = float(line[60:66])
        except (ValueError, IndexError): bfac = 0.0
        is_h = atom.startswith("H") or (len(atom) > 1 and atom[1] == "H" and atom[0].isdigit())
        key = (chain, resnum, icode)
        if key not in residues:
            residues[key] = {"resname": line[17:20].strip(), "atoms": [], "ca": None, "cb": None, "sc": []}
            order.append(key)
        r = residues[key]
        r["atoms"].append((atom, x, y, z, bfac, is_h))
        if   atom == "CA": r["ca"] = (x, y, z)
        elif atom == "CB": r["cb"] = (x, y, z)
        if atom not in {"N","CA","C","O","OXT"} and not is_h:
            r["sc"].append((x, y, z))
    return order, residues

In [4]:
def compute_features(order, residues):
    keys = [k for k in order if residues[k]["ca"] is not None]
    n = len(keys)
    if n == 0:
        return [], np.zeros((0, NUM_FEATURES), np.float32)

    ca = np.array([residues[k]["ca"] for k in keys], np.float32)
    cb = np.array([residues[k]["cb"] or residues[k]["ca"] for k in keys], np.float32)
    side_dir = cb - ca
    side_dir /= (np.linalg.norm(side_dir, axis=1, keepdims=True) + 1e-6)

    centroid = ca.mean(0)
    max_extent = np.linalg.norm(ca - centroid, axis=1).max() + 1e-6

    sq = (ca * ca).sum(1)
    dist = np.sqrt(np.maximum(sq[:,None] + sq[None,:] - 2*(ca @ ca.T), 0.0)).astype(np.float32)
    np.fill_diagonal(dist, np.inf)

    n6,n8,n10,n12,n15 = ((dist<r).sum(1) for r in (6.,8.,10.,12.,15.))

    k = min(10, n-1) if n > 1 else 1
    if k > 0:
        nn = np.take_along_axis(dist, np.argpartition(dist, k, axis=1)[:, :k], axis=1)
        mean_nn, std_nn = nn.mean(1), nn.std(1)
    else:
        mean_nn = std_nn = np.zeros(n, np.float32)

    mask13, mask10 = dist < 13.0, dist < 10.0
    up=np.zeros(n,np.float32); down=np.zeros(n,np.float32); co_plane=np.zeros(n,np.float32)
    same=np.zeros(n,np.float32); opp=np.zeros(n,np.float32); mean_dir=np.zeros((n,3),np.float32)
    for s in range(0, n, 256):                          # chunk to bound memory on big proteins
        e = min(s+256, n)
        rel = (ca[None,s:e,:] - ca[:,None,:]).transpose(1,0,2)
        proj = (rel * side_dir[s:e,None,:]).sum(2)
        m13, m10 = mask13[s:e], mask10[s:e]
        up[s:e]       = ((proj > 0) & m13).sum(1)
        down[s:e]     = ((proj < 0) & m13).sum(1)
        co_plane[s:e] = ((np.abs(proj) < 4) & m10).sum(1)
        same[s:e]     = ((proj > 1) & m10).sum(1)
        opp[s:e]      = ((proj < -1) & m10).sum(1)
        safe = np.where(m10, dist[s:e], 1.0).astype(np.float32)
        mean_dir[s:e] = ((rel / safe[...,None]) * m10[...,None]).sum(1)

    hse_ratio     = (up - down) / (up + down + 1.0)
    pocket_open   = (same - opp) / (same + opp + 1.0)
    pocket_concav = co_plane / (mask10.sum(1).astype(np.float32) + 1.0)

    surface_mask = n12 <= np.median(n12)
    depth_proxy  = dist[:, surface_mask].min(1) if surface_mask.any() else np.zeros(n, np.float32)

    d_cent = np.linalg.norm(ca - centroid, axis=1)
    ca_z   = (ca[:,2] - ca[:,2].mean()) / (ca[:,2].std() + 1e-6)
    sa_proxy = 1.0 / (1.0 + n12.astype(np.float32)/10.0)

    counts10 = mask10.sum(1).astype(np.float32)
    mean_dir /= (counts10[:,None] + 1e-3)
    anisotropy = np.linalg.norm(mean_dir, axis=1).astype(np.float32)
    mean_dir_n = mean_dir / (np.linalg.norm(mean_dir, axis=1, keepdims=True) + 1e-6)
    rel_burial = (mean_dir_n * side_dir).sum(1).astype(np.float32)

    bfac_mean=np.zeros(n,np.float32); bfac_std=np.zeros(n,np.float32); sc_extent=np.zeros(n,np.float32)
    for i, key in enumerate(keys):
        bvals = [a[4] for a in residues[key]["atoms"] if not a[5]]
        if bvals: bfac_mean[i], bfac_std[i] = np.mean(bvals), np.std(bvals)
        sc = residues[key]["sc"]
        if sc: sc_extent[i] = np.linalg.norm(np.asarray(sc, np.float32) - ca[i], axis=1).max()

    aa_onehot = np.zeros((n, len(AA_ORDER)+1), np.float32)
    prop = np.zeros((n, len(PROP_NAMES)), np.float32)
    for i, key in enumerate(keys):
        aa = AA3_TO_1.get(residues[key]["resname"], "X")
        aa_onehot[i, AA_INDEX.get(aa, len(AA_ORDER))] = 1.0
        prop[i] = [KD.get(aa,0), VOL.get(aa,0), CHARGE.get(aa,0), aa in POLAR, aa in AROMATIC,
                   aa in ALIPHATIC, aa in HBOND_DONOR, aa in HBOND_ACCEPTOR, aa == "G"]

    geom = np.stack([n6,n8,n10,n12,n15, d_cent, d_cent/max_extent, depth_proxy,
        mean_nn, std_nn, up, down, hse_ratio, ca_z, bfac_mean, bfac_std, sc_extent,
        pocket_open, pocket_concav, sa_proxy, anisotropy, rel_burial], 1).astype(np.float32)

    prop_sub = prop[:, [PROP_NAMES.index(p) for p in NEIGH_AGG_PROPS]]
    neigh = []
    for radius in NEIGH_RADII:
        m = dist < radius
        neigh.append((m.astype(np.float32) @ prop_sub) / (m.sum(1).astype(np.float32)[:,None] + 1e-3))
    X = np.concatenate([aa_onehot, prop, geom, np.concatenate(neigh, 1)], 1).astype(np.float32)

    res_ids = [f"{c}_{r}_{ic}" if ic else f"{c}_{r}" for (c, r, ic) in keys]
    return res_ids, X

def featurize(pdb_bytes):
    return compute_features(*parse_pdb_atoms(pdb_bytes))

# demo on one protein
rid, X0 = featurize(read_pdb_bytes(TRAIN_DIR, 0))
print(f"protein 0: {len(rid)} residues, feature matrix {X0.shape}")
print("first residue id:", rid[0], "| feature vector head:", np.round(X0[0, 21:29], 2))

protein 0: 501 residues, feature matrix (501, 66)
first residue id: A_5 | feature vector head: [ -4.5 173.4   1.    1.    0.    0.    1.    0. ]


## Build the training matrix

In [5]:
FULL_RUN = False
TRAIN_CACHE = os.path.join(CACHE_DIR, "train_features.npz")

def featurize_ids(folder, pids):
    pid_a, rid_a, X_a = [], [], []
    for p in pids:
        b = read_pdb_bytes(folder, p)
        if b is None: continue
        rids, X = featurize(b)
        if rids:
            pid_a.append(np.full(len(rids), p, np.int32)); rid_a.append(np.asarray(rids, object)); X_a.append(X)
    return np.concatenate(pid_a), np.concatenate(rid_a), np.concatenate(X_a, 0)

if os.path.exists(TRAIN_CACHE):
    d = np.load(TRAIN_CACHE, allow_pickle=True)
    pid, resid, X = d["pid"], d["resid"].astype(str), d["X"]
    print(f"loaded cache: {X.shape[0]:,} residues over {len(np.unique(pid)):,} proteins")
else:
    demo_pids = sorted(labels["id"])[: (len(labels) if FULL_RUN else 300)]
    print(f"no cache — featurising {len(demo_pids)} proteins live "
          f"({'FULL' if FULL_RUN else 'subset demo'})...")
    t0 = time.time(); pid, resid, X = featurize_ids(TRAIN_DIR, demo_pids)
    print(f"  {X.shape[0]:,} residues in {time.time()-t0:.0f}s")

loaded cache: 7,454,527 residues over 16,379 proteins


## Train the gradient-boosted classifier

In [6]:
RETRAIN = False                        # True = fit from scratch; False = load shipped model
MODEL_PATH = os.path.join(MODEL_DIR, "model.pkl")

pos = {int(r.id): set(str(r.resid).split()) for r in labels.itertuples()}
y = np.fromiter(((rid in pos.get(int(p), ())) for p, rid in zip(pid, resid)),
                dtype=np.int8, count=len(pid))
print(f"positive rate: {y.mean():.3f}  ({int(y.sum()):,} / {len(y):,})")

uniq = np.unique(pid); rng.shuffle(uniq)
val_ids = set(uniq[: max(1, int(0.1*len(uniq)))].tolist())
is_val  = np.fromiter((int(p) in val_ids for p in pid), bool, len(pid))

if RETRAIN or not os.path.exists(MODEL_PATH):
    clf = HistGradientBoostingClassifier(max_iter=800, learning_rate=0.05, max_leaf_nodes=63,
        min_samples_leaf=80, l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=30, class_weight="balanced", random_state=SEED)
    t0 = time.time(); clf.fit(X[~is_val], y[~is_val]); print(f"trained in {time.time()-t0:.0f}s")
    pv = clf.predict_proba(X[is_val])[:, 1]
    best_t, best_j = max(((float(jaccard_score(y[is_val], (pv>=t))), t)
                          for t in np.linspace(0.05, 0.95, 91)), key=lambda z: z[0])[::-1]
    pickle.dump({"model": clf, "threshold": best_t, "val_score": best_j, "feature_cols": FEATURE_COLS},
                open(MODEL_PATH, "wb"))
    print(f"val flat Jaccard {best_j:.4f} at threshold {best_t:.3f}")
else:
    payload = pickle.load(open(MODEL_PATH, "rb"))
    clf, best_t = payload["model"], payload["threshold"]
    print(f"loaded shipped model — threshold {best_t:.3f}, stored val Jaccard {payload['val_score']:.4f}")

pv = clf.predict_proba(X[is_val])[:, 1]
print(f"val AUC {roc_auc_score(y[is_val], pv):.3f} | AP {average_precision_score(y[is_val], pv):.3f}")

positive rate: 0.110  (819,331 / 7,454,527)


/opt/anaconda3/envs/mlpc2026/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator _BinMapper from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/envs/mlpc2026/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator HistGradientBoostingClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loaded shipped model — threshold 0.725, stored val Jaccard 0.3678
val AUC 0.877 | AP 0.574


## Spatial post-processing

* **dilate** — promote an uncertain residue (`prob ≥ t_soft`) to positive if it
  lies within `radius` of a confident one (`prob ≥ t_hard`); fills pocket rims.
* **prune** — drop connected components of positives smaller than `min_size`;
  removes isolated single-residue false positives.

Sweep these on the held-out proteins against the flat Jaccard and keep the
best. The gain is small but bootstrap validated. 

In [8]:
def _pdist(c):
    sq = (c*c).sum(1)
    return np.sqrt(np.maximum(sq[:,None]+sq[None,:]-2*(c@c.T), 0.0)).astype(np.float32)

def dilate(coords, prob, t_hard, t_soft, radius, dist=None):
    hard, soft = prob >= t_hard, prob >= t_soft
    sel = hard.copy()
    if not (hard.any() and soft.any()): return sel
    if dist is None: dist = _pdist(coords)
    hi = np.where(hard)[0]
    for j in np.where(soft & ~hard)[0]:
        if (dist[j, hi] < radius).any(): sel[j] = True
    return sel

def prune(coords, mask, radius, min_size, dist=None):
    if min_size <= 1 or mask.sum() == 0: return mask.copy()
    idx = np.where(mask)[0]
    D = _pdist(coords[idx]) if dist is None else dist[np.ix_(idx, idx)]
    adj = D < radius; np.fill_diagonal(adj, False)
    seen = np.zeros(len(idx), bool); keep = np.zeros(len(idx), bool)
    for s in range(len(idx)):
        if seen[s]: continue
        stack=[s]; seen[s]=True; comp=[s]
        while stack:
            u = stack.pop()
            for v in np.where(adj[u] & ~seen)[0]:
                seen[v]=True; comp.append(v); stack.append(v)
        if len(comp) >= min_size: keep[comp] = True
    out = mask.copy(); out[idx] = keep; return out

def apply_strategy(coords, prob, strat, p, dist=None):
    if strat == "threshold":    return prob >= p["t"]
    if strat == "dilate":       return dilate(coords, prob, p["t_hard"], p["t_soft"], p["radius"], dist)
    if strat == "dilate_prune":
        m = dilate(coords, prob, p["t_hard"], p["t_soft"], p["dilate_radius"], dist)
        return prune(coords, m, p["prune_radius"], p["min_size"], dist)
    raise ValueError(strat)

# CA coords for the held-out proteins
def val_coords(pid_val):
    order, res = parse_pdb_atoms(read_pdb_bytes(TRAIN_DIR, pid_val))
    keys = [k for k in order if res[k]["ca"] is not None]
    ids  = [f"{c}_{r}_{ic}" if ic else f"{c}_{r}" for (c,r,ic) in keys]
    cmap = {i: res[k]["ca"] for i, k in zip(ids, keys)}
    return ids, cmap

# group val probabilities per protein
val_pids = sorted(val_ids)
grp = {p: ([], []) for p in val_pids}
for p, r, q in zip(pid[is_val], resid[is_val], pv):
    grp[int(p)][0].append(r); grp[int(p)][1].append(q)

coords_by, dist_by, ytrue_by, prob_by, rid_by = {}, {}, {}, {}, {}
for p in val_pids:
    rids, probs = grp[p]
    rids = np.asarray(rids); probs = np.asarray(probs, np.float32)
    ids, cmap = val_coords(p)
    coords = np.array([cmap[r] for r in rids], np.float32)
    coords_by[p]=coords; dist_by[p]=_pdist(coords)
    prob_by[p]=probs; rid_by[p]=rids
    tset = pos.get(p, set())
    ytrue_by[p] = np.fromiter((r in tset for r in rids), np.int8, len(rids))

def flat_jaccard(strat, params):
    yt = np.concatenate([ytrue_by[p] for p in val_pids])
    yp = np.concatenate([apply_strategy(coords_by[p], prob_by[p], strat, params, dist_by[p]).astype(np.int8)
                         for p in val_pids])
    return jaccard_score(yt, yp)

base = flat_jaccard("threshold", {"t": best_t})
print(f"baseline threshold {best_t:.3f}: flat Jaccard {base:.4f}")

best = ("threshold", {"t": best_t}, base)
for th in [0.7, best_t, 0.75, 0.8]:
    for ts in [0.4, 0.5, 0.6]:
        if ts >= th: continue
        for rad in [6., 8.]:
            s = flat_jaccard("dilate", {"t_hard":th,"t_soft":ts,"radius":rad})
            if s > best[2]: best = ("dilate", {"t_hard":th,"t_soft":ts,"radius":rad}, s)
for pr in [8.]:
    for ms in [3, 5, 6]:
        p = dict(best[1])
        if best[0] != "dilate": continue
        params = {"dilate_radius":p["radius"],"t_hard":p["t_hard"],"t_soft":p["t_soft"],
                  "prune_radius":pr,"min_size":ms}
        s = flat_jaccard("dilate_prune", params)
        if s > best[2]: best = ("dilate_prune", params, s)

print(f"best strategy: {best[0]} {best[1]}  ->  flat Jaccard {best[2]:.4f}")

pickle.dump({"winner_strategy":best[0], "winner_params":best[1], "winner_score":best[2],
             "baseline_score":base}, open(os.path.join(CACHE_DIR,"postprocess_choice_demo.pkl"), "wb"))
CANON = os.path.join(MODEL_DIR, "postprocess_choice.pkl")
if os.path.exists(CANON):
    canon = pickle.load(open(CANON, "rb"))
    print(f"canonical full sweep: {canon['winner_strategy']} {canon['winner_params']}"
          f"  ->  flat Jaccard {canon['winner_score']:.4f}")
    best = (canon["winner_strategy"], canon["winner_params"], canon["winner_score"])

baseline threshold 0.725: flat Jaccard 0.3678
best strategy: dilate_prune {'dilate_radius': 6.0, 't_hard': 0.8, 't_soft': 0.6, 'prune_radius': 8.0, 'min_size': 3}  ->  flat Jaccard 0.3703
canonical full sweep: dilate_prune {'dilate_radius': 6.0, 't_hard': 0.8, 't_soft': 0.6, 'prune_radius': 8.0, 'min_size': 6}  ->  flat Jaccard 0.3706


## Predict on the test set

In [10]:
WRITE_SUBMISSION = False               # flip to (re)generate the file
TEST_CACHE = os.path.join(CACHE_DIR, "test_features.npz")
SUB_PATH   = os.path.join(ROOT, "01_gbm_handcrafted", "submission_v2_postprocessed.csv")
strat, sp = best[0], best[1]

def predict_protein(pid_test, coords, res_ids, Xt):
    prob = clf.predict_proba(Xt)[:, 1]
    mask = apply_strategy(coords, prob, strat, sp)
    return " ".join(r for r, m in zip(res_ids, mask) if m)

if WRITE_SUBMISSION:
    rows = []
    ids = sorted(int(f[:-len("_protein.pdb")]) for f in os.listdir(TEST_DIR) if f.endswith("_protein.pdb"))
    t0 = time.time()
    for i, p in enumerate(ids):
        order, res = parse_pdb_atoms(read_pdb_bytes(TEST_DIR, p))
        rids, Xt = compute_features(order, res)
        if not rids: rows.append({"id": p, "prediction": ""}); continue
        keys = [k for k in order if res[k]["ca"] is not None]
        coords = np.array([res[k]["ca"] for k in keys], np.float32)
        rows.append({"id": p, "prediction": predict_protein(p, coords, rids, Xt)})
        if (i+1) % 500 == 0: print(f"  {i+1}/{len(ids)}  ({(i+1)/(time.time()-t0):.1f}/s)")
    pd.DataFrame(rows).to_csv(SUB_PATH, index=False)
    print("wrote", SUB_PATH)
else:
    print("WRITE_SUBMISSION is False — using the committed submission file.")
    print("existing:", os.path.basename(SUB_PATH), "->", os.path.exists(SUB_PATH))

WRITE_SUBMISSION is False — using the committed submission file.
existing: submission_v2_postprocessed.csv -> True


## Local evaluation| mirrors `metric.py`

In [11]:
def local_flat_jaccard(submission_csv, pids):
    sub = pd.read_csv(submission_csv)
    sub["prediction"] = sub["prediction"].fillna("")
    pred = {int(r.id): set(str(r.prediction).split()) for r in sub.itertuples()}
    yt, yp = [], []
    for p in pids:
        order, res = parse_pdb_atoms(read_pdb_bytes(TRAIN_DIR, p))
        ids = [f"{c}_{r}_{ic}" if ic else f"{c}_{r}"
               for (c,r,ic) in order if res[(c,r,ic)]["ca"] is not None]
        tset, pset = pos.get(p, set()), pred.get(p, set())
        for i in ids:
            yt.append(1 if i in tset else 0); yp.append(1 if i in pset else 0)
    return jaccard_score(yt, yp)

# score the winning strategy on a slice of held-out proteins as a demo
demo = val_pids[:150]
demo_rows = []
for p in demo:
    demo_rows.append({"id": p, "prediction":
        " ".join(r for r, m in zip(rid_by[p],
                 apply_strategy(coords_by[p], prob_by[p], strat, sp, dist_by[p])) if m)})
tmp = os.path.join(CACHE_DIR, "_demo_sub.csv"); pd.DataFrame(demo_rows).to_csv(tmp, index=False)
print(f"held-out flat Jaccard on {len(demo)} demo proteins: {local_flat_jaccard(tmp, demo):.4f}")

held-out flat Jaccard on 150 demo proteins: 0.3318
